# Comma Video Compression — Production Training (Colab)

Run the validated winner architecture (CONFIG=B + tuned HPs) on a Colab A100 for the full submission.

**Plan:**
1. Verify A100 is assigned (restart runtime if T4)
2. Clone repo + install deps
3. Set config variables (the cell with all tweakable knobs)
4. Run training (~6-11 hours depending on bs)
5. Save model weights + run.log back to Google Drive

**Cost estimate:** ~$10-15 in Colab compute units for an A100 run.

## 1. Verify GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
# If this isn't 'NVIDIA A100' (or H100), go to Runtime → Change runtime type → A100 GPU.

## 2. Mount Google Drive (for output storage)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
OUTPUT_DIR = '/content/drive/MyDrive/comma_compression_runs'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'Outputs will save to: {OUTPUT_DIR}')

## 3. Clone repo

Make sure you've pushed `autoresearch/apr29` (or whichever branch) to a private GitHub. The repo includes `videos/0.mkv` and `models/*.safetensors` (~130MB), so cloning gives you everything in one shot.

In [ ]:
# ─── Set these to your repo ───
REPO_URL = 'https://github.com/YOUR_USERNAME/comma_video_compression_challenge.git'
BRANCH   = 'autoresearch/apr29'
GH_TOKEN = ''  # Personal Access Token if private. Leave empty if public.

# Clone
!cd /content && rm -rf comma_video_compression_challenge
if GH_TOKEN:
    url_with_token = REPO_URL.replace('https://', f'https://{GH_TOKEN}@')
    !cd /content && git clone --branch {BRANCH} --single-branch {url_with_token}
else:
    !cd /content && git clone --branch {BRANCH} --single-branch {REPO_URL}

%cd /content/comma_video_compression_challenge
!ls -la videos/ models/

## 4. Install dependencies

In [ ]:
# Most of these are pre-installed on Colab; only missing ones get pulled.
!pip install -q einops safetensors timm segmentation_models_pytorch av brotli
!python -c "import torch; print('torch', torch.__version__); print('cuda', torch.cuda.is_available())"

## 5. ⚙️ CONFIG — all tweakable knobs in one place

Edit this cell to change behavior. Defaults below = **validated winner from Phase 1+2 HP tuning**:
- CONFIG=B (boundary-weighted focal + Lion) won at full 1hr validation
- COSINE_LR=1 won at HP tuning (E3/E6 both improved)
- EMA_DECAY=0.999 (extrapolated for 19h scale; EMA 0.99 won at 30min but at 19h scale, 0.999 averages ~10% of training, the right window)
- GRAD_CLIP=0.5 (E4 confirmed loosening hurts)

**For T4 smoke test** (16GB VRAM): set `BATCH_SIZE = 2` and `TRAIN_BUDGET_SEC = 300` to verify pipeline runs end-to-end. T4 is too slow for full submission (~50hr equivalent of 3090).</cell>

In [ ]:
# ─── Architecture (validated winner) ───
CONFIG       = 'B'      # boundary-weighted focal + Lion
FULL_DATA    = 1        # 500 train / 100 val from all 600 pairs

# ─── Training budget (wallclock seconds) ───
# T4 smoke test:  300  (5 min — just verify pipeline)
# A100 full run:  30000 (8.3h — recommended for production)
# A100 longer:    40000 (11h — safer if A100 is slow variant)
TRAIN_BUDGET_SEC = 300  # ⚠️ START WITH 300 FOR T4 SMOKE TEST, then bump to 30000 for A100

# ─── Tuned hyperparameters (validated from HP tuning) ───
EMA_DECAY    = 0.999    # 0.999 for 19h scale (averages ~10% of training)
COSINE_LR    = 1        # cosine decay within each stage (validated win)
GRAD_CLIP    = 0.5      # default 0.5 (validated; loosening to 1.0 hurt)

# ─── Periodic checkpoint to Drive (Colab disconnect insurance) ───
# Every N seconds during joint stage, saves EMA state to {SAVE_MODEL_PATH}.ckpt
# Set to 1800 (30min) for production. 0 = disabled.
CHECKPOINT_INTERVAL_SEC = 1800

# ─── Batch size (CAREFUL — affects memory + training dynamics) ───
# T4 16GB:    bs=2 (fits)
# 3090 24GB:  bs=4 (proven, what we validated at)
# A100 40GB:  bs=4 (safe) or bs=6 (likely OK, ~1.4× faster)
# A100 80GB:  bs=8 (much faster)
BATCH_SIZE   = 2        # ⚠️ START WITH 2 FOR T4 SMOKE TEST, then 4 (safe) or 6 for A100

# ─── Where to save outputs ───
RUN_NAME     = f'configB_ema{EMA_DECAY}_cosine{COSINE_LR}_bs{BATCH_SIZE}'
RUN_DIR      = f'{OUTPUT_DIR}/{RUN_NAME}'
os.makedirs(RUN_DIR, exist_ok=True)
print(f'\nWill save to: {RUN_DIR}')
print(f'Effective config: CONFIG={CONFIG} FULL_DATA={FULL_DATA} BUDGET={TRAIN_BUDGET_SEC}s ({TRAIN_BUDGET_SEC/3600:.1f}h)')
print(f'                  EMA={EMA_DECAY} COSINE_LR={COSINE_LR} GRAD_CLIP={GRAD_CLIP} BS={BATCH_SIZE}')
print(f'                  CHECKPOINT_INTERVAL={CHECKPOINT_INTERVAL_SEC}s ({CHECKPOINT_INTERVAL_SEC/60:.0f} min)')

## 6. Apply BATCH_SIZE override to train.py

(BATCH_SIZE is hardcoded in `train.py` rather than env-var driven, so we patch it inline before launching.)

In [ ]:
import re
with open('autoresearch/train.py', 'r') as f:
    src = f.read()
src_new = re.sub(r'^BATCH_SIZE\s*=\s*\d+', f'BATCH_SIZE    = {BATCH_SIZE}', src, count=1, flags=re.M)
with open('autoresearch/train.py', 'w') as f:
    f.write(src_new)
!grep -m1 '^BATCH_SIZE' autoresearch/train.py

## 7. (Optional) Smoke test — 5 min to verify everything works

Skip if confident. If you change BS, definitely run this first to catch OOM.

In [ ]:
import os, subprocess, time
env = os.environ.copy()
env.update({
    'PYTHONUNBUFFERED': '1',
    'CONFIG': CONFIG,
    'FULL_DATA': str(FULL_DATA),
    'TRAIN_BUDGET_SEC_OVERRIDE': str(TRAIN_BUDGET_SEC),  # uses your CONFIG cell value (300 for T4)
    'EMA_DECAY': str(EMA_DECAY),
    'COSINE_LR': str(COSINE_LR),
    'GRAD_CLIP_OVERRIDE': str(GRAD_CLIP),
    'SAVE_MODEL_PATH': f'{RUN_DIR}/gen_smoke.pt',  # different file so it doesn't clobber the real run
})
print(f'Smoke test ({TRAIN_BUDGET_SEC}s, bs={BATCH_SIZE})...')
print('Watch for: data loads, all 3 stages run, score prints, gen_smoke.pt saved.')
subprocess.run(['python', 'autoresearch/train.py'], env=env, check=True)
print(f'\n✅ Smoke test done. Now bump TRAIN_BUDGET_SEC + BATCH_SIZE in CONFIG cell, switch runtime to A100, and run the full training cell below.')

## 8. Full training run

Runs in foreground so you can watch logs stream. Will take TRAIN_BUDGET_SEC + a few minutes for eval.

In [ ]:
import os, subprocess, time
env = os.environ.copy()
env.update({
    'PYTHONUNBUFFERED': '1',
    'CONFIG': CONFIG,
    'FULL_DATA': str(FULL_DATA),
    'TRAIN_BUDGET_SEC_OVERRIDE': str(TRAIN_BUDGET_SEC),
    'EMA_DECAY': str(EMA_DECAY),
    'COSINE_LR': str(COSINE_LR),
    'GRAD_CLIP_OVERRIDE': str(GRAD_CLIP),
    'SAVE_MODEL_PATH': f'{RUN_DIR}/gen.pt',
    'CHECKPOINT_INTERVAL_SEC': str(CHECKPOINT_INTERVAL_SEC),
})
log_path = f'{RUN_DIR}/run.log'
print(f'Starting training. Log → {log_path}')
print(f'Final model → {RUN_DIR}/gen.pt')
print(f'Periodic checkpoints → {RUN_DIR}/gen.pt.ckpt (every {CHECKPOINT_INTERVAL_SEC}s during joint)')
print(f'Wallclock budget: {TRAIN_BUDGET_SEC}s ({TRAIN_BUDGET_SEC/3600:.1f}h)')
print('═'*60)
t0 = time.time()
with open(log_path, 'w') as f:
    proc = subprocess.Popen(['python', 'autoresearch/train.py'], env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end='')
        f.write(line); f.flush()
    proc.wait()
print('═'*60)
print(f'Done in {time.time()-t0:.1f}s. Exit code: {proc.returncode}')

## 9. Save model weights to Drive

Pickles the trained generator's state_dict (with FP4-quantized weights applied).

In [ ]:
import shutil
# train.py wrote the EMA-averaged state_dict to {RUN_DIR}/gen.pt during eval
# (because we passed SAVE_MODEL_PATH in the env). Also save code + log for reproducibility.
shutil.copy('autoresearch/train.py', f'{RUN_DIR}/train.py')
shutil.copy('autoresearch/prepare.py', f'{RUN_DIR}/prepare.py')
print(f'Saved to {RUN_DIR}:')
!ls -la {RUN_DIR}
print('\nFinal scores from run.log:')
!grep -E '^score:|^seg_term:|^pose_term:|^rate_term:|^model_bytes:|^total_epochs:|^saved_model:' {RUN_DIR}/run.log

## 10. (TODO) Build the submission archive

The contest expects a specific format — model weights + mask video + pose floats packed together. Need to add a cell that does this once we know the submission spec.

Note: train.py currently only prints the score; it doesn't dump the trained model. To save the model for submission, add a `torch.save(gen.state_dict(), 'gen.pt')` at the end of train.py (after eval), then copy to Drive.